# Hit Score experimental

En este notebook se construye una métrica experimental llamada `Hit Score`.

El objetivo del `Hit Score` es resumir qué tan alineada está una canción con el perfil acústico observado en canciones populares dentro del dataset.

A partir del análisis exploratorio previo, se identificaron algunas variables asociadas con mayor popularidad. Las canciones populares tienden a presentar mayor `danceability`, mayor `energy`, mayor `loudness`, menor `acousticness` y menor `instrumentalness`.

Por esta razón, el `Hit Score` se construirá combinando estas variables acústicas después de normalizarlas a una escala común.

Es importante aclarar que el `Hit Score` no representa una predicción real de éxito comercial ni una probabilidad de que una canción se vuelva popular. Se trata de una métrica académica y exploratoria cuyo propósito es demostrar cómo se pueden transformar datos musicales masivos en un indicador interpretable.

Esta métrica permitirá ordenar canciones según su similitud con el perfil de los hits y, posteriormente, buscar canciones poco populares que tengan características acústicas parecidas a canciones exitosas.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from functools import reduce

In [2]:
spark = (
    SparkSession.builder
    .appName("HitMakerAI_HitScore")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/04 00:39:50 WARN Utils: Your hostname, debian, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp3s0)
26/05/04 00:39:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/04 00:39:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/04 00:39:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
tracks_base = spark.read.parquet("../processed/tracks_base.parquet")

tracks_base.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- explicit: integer (nullable = true)
 |-- artists: string (nullable = true)
 |-- id_artists: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: integer (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- release_decade: integer (nullable = true)
 |-- duration_min: double (nullable = true)
 |-- is_hit: integer (nullable = true)
 |--

In [4]:
#verificamos que cargó de forma correcta
tracks_base.select(
    F.count("*").alias("total_filas"),
    F.countDistinct("id").alias("ids_unicos")
).show()

[Stage 1:>                                                          (0 + 8) / 8]

+-----------+----------+
|total_filas|ids_unicos|
+-----------+----------+
|     586264|    586264|
+-----------+----------+



Ahora, por el Análisis exploratorio que realizamos anteriormente, nos dimos cuenta que entre mayor era el "acousticness" y el "instrumentalness" **menos popular es**, mientras que entre mayor era "danceability","energy", "loudness" **más popular es**. Por lo que lo separaremos entre positivo y negativo:

In [5]:
positive_features = [
    "danceability",
    "energy",
    "loudness"
]

negative_features = [
    "acousticness",
    "instrumentalness"
]

selected_features = positive_features + negative_features

## Normalización

la mayoria de las variables ya estan en un rango de 0 y 2. Pero hay algunas que todavía no están normalizadas como loudness

### Calcular mínimos y máximos para normalización

In [7]:
min_max_exprs = []

for feature in selected_features:
    min_max_exprs.append(F.min(feature).alias(f"min_{feature}"))
    min_max_exprs.append(F.max(feature).alias(f"max_{feature}"))

feature_stats = tracks_base.agg(*min_max_exprs)

feature_stats.show()

+----------------+----------------+----------+----------+------------+------------+----------------+----------------+--------------------+--------------------+
|min_danceability|max_danceability|min_energy|max_energy|min_loudness|max_loudness|min_acousticness|max_acousticness|min_instrumentalness|max_instrumentalness|
+----------------+----------------+----------+----------+------------+------------+----------------+----------------+--------------------+--------------------+
|          0.0532|           0.991|       0.0|       1.0|       -55.0|       5.376|             0.0|           0.996|                 0.0|                 1.0|
+----------------+----------------+----------+----------+------------+------------+----------------+----------------+--------------------+--------------------+



In [8]:
tracks_with_stats = tracks_base.crossJoin(feature_stats)

tracks_scored = tracks_with_stats

for feature in selected_features:
    min_col = F.col(f"min_{feature}")
    max_col = F.col(f"max_{feature}")
    
    tracks_scored = tracks_scored.withColumn(
        f"{feature}_norm",
        F.when(
            max_col != min_col,
            (F.col(feature) - min_col) / (max_col - min_col)
        ).otherwise(F.lit(0.0))
    )

Verificamos algunas columnas normalizadas

In [10]:
tracks_scored.select(
    "name",
    "popularity",
    "danceability",
    "danceability_norm",
    "energy",
    "energy_norm",
    "loudness",
    "loudness_norm",
    "acousticness",
    "acousticness_norm",
    "instrumentalness",
    "instrumentalness_norm"
).show(10)

+--------------------+----------+------------+-------------------+------+-----------+--------+------------------+------------+--------------------+----------------+---------------------+
|                name|popularity|danceability|  danceability_norm|energy|energy_norm|loudness|     loudness_norm|acousticness|   acousticness_norm|instrumentalness|instrumentalness_norm|
+--------------------+----------+------------+-------------------+------+-----------+--------+------------------+------------+--------------------+----------------+---------------------+
|Pumping On Your S...|        50|       0.461|  0.434847515461719| 0.904|      0.904|  -4.789|0.8316383993639856|       0.011|0.011044176706827308|          0.0852|               0.0852|
|  Mujhe Raat Din Bas|        50|       0.614| 0.5979953081680529| 0.469|      0.469| -13.124|0.6935868557042534|         0.4| 0.40160642570281124|         3.93E-6|              3.93E-6|
| Message in a Bottle|        50|       0.324|0.28876092983578594

Para las variables positivas usamos directamente la versión normalizada.

Para las variables negativas usamos: $1 - variable Normalizada$

In [11]:
tracks_scored = (
    tracks_scored
    .withColumn("danceability_score", F.col("danceability_norm"))
    .withColumn("energy_score", F.col("energy_norm"))
    .withColumn("loudness_score", F.col("loudness_norm"))
    .withColumn("low_acousticness_score", 1 - F.col("acousticness_norm"))
    .withColumn("low_instrumentalness_score", 1 - F.col("instrumentalness_norm"))
)

## Construcción del HitScore

Primera versión:

In [12]:
score_components = [
    F.col("danceability_score"),
    F.col("energy_score"),
    F.col("loudness_score"),
    F.col("low_acousticness_score"),
    F.col("low_instrumentalness_score")
]

tracks_scored = (
    tracks_scored
    .withColumn(
        "hit_score_raw",
        reduce(lambda a, b: a + b, score_components) / len(score_components)
    )
    .withColumn(
        "hit_score",
        F.round(F.col("hit_score_raw") * 100, 2)
    )
)

Revisamos canciones con mayor hit_score:

In [14]:
tracks_scored.select(
    "name",
    "artists",
    "release_year",
    "popularity",
    "is_hit",
    "hit_score",
    "danceability",
    "energy",
    "loudness",
    "acousticness",
    "instrumentalness"
).orderBy(F.desc("hit_score")).show(100)

+--------------------+--------------------+------------+----------+------+---------+------------+------+--------+------------+----------------+
|                name|             artists|release_year|popularity|is_hit|hit_score|danceability|energy|loudness|acousticness|instrumentalness|
+--------------------+--------------------+------------+----------+------+---------+------------+------+--------+------------+----------------+
| Cerol na Mão Techno|    ['Furacão 2000']|        2000|        40|     0|    95.48|       0.964|  0.96|  -3.066|      0.0172|         1.37E-6|
|Išvažiuosiu Prie ...|['Džordana Butkutė']|        2001|        19|     0|    95.15|       0.954| 0.991|  -5.768|     8.55E-4|          0.0085|
|   Critical Beatdown|["Ultramagnetic M...|        1988|        36|     0|    94.87|       0.933| 0.959|  -3.682|     0.00337|         1.23E-6|
|    Dança da Motinha|            ['Beth']|        2000|        43|     0|    94.69|       0.973| 0.957|  -5.969|      0.0105|         0

Clasificar el HitScore

In [16]:
tracks_scored = tracks_scored.withColumn(
    "hit_score_class",
    F.when(F.col("hit_score") < 40, "bajo")
     .when(F.col("hit_score") < 70, "medio")
     .otherwise("alto")
)

Distribución del Hit Score

In [17]:
hit_score_class_dist = (
    tracks_scored
    .groupBy("hit_score_class")
    .agg(F.count("*").alias("num_canciones"))
    .orderBy("hit_score_class")
)

hit_score_class_dist.show(truncate=False)

+---------------+-------------+
|hit_score_class|num_canciones|
+---------------+-------------+
|alto           |266112       |
|bajo           |45494        |
|medio          |274658       |
+---------------+-------------+



Validar si el score se relaciona con popularidad

In [18]:
tracks_scored.select(
    F.round(F.corr("hit_score", "popularity"), 4).alias("corr_hit_score_popularity")
).show()

+-------------------------+
|corr_hit_score_popularity|
+-------------------------+
|                   0.4115|
+-------------------------+



**Relación entre Hit Score y popularidad**

Para evaluar si el `Hit Score` tiene sentido como métrica exploratoria, se calculó la correlación entre `hit_score` y `popularity`.

El resultado obtenido fue una correlación positiva de aproximadamente `0.4115`. Esto indica una relación positiva moderada: en general, las canciones con mayor `Hit Score` tienden a tener mayor popularidad.

Este resultado es relevante porque el `Hit Score` no utiliza directamente la variable `popularity` para calcularse. En cambio, se construye a partir de características acústicas seleccionadas durante el análisis exploratorio.

Por lo tanto, la correlación positiva sugiere que el score captura parcialmente un perfil acústico asociado con canciones populares.

Sin embargo, la correlación no es perfecta, lo cual es esperado. La popularidad musical depende de muchos factores que no están incluidos en esta métrica, como el artista, el contexto cultural, el idioma, la letra, la promoción, las playlists o tendencias externas.

Así, el `Hit Score` debe interpretarse como una métrica experimental de similitud acústica con canciones populares, no como una predicción definitiva de éxito.

*Validación por rangos*:
queremos ver ahí es si conforme sube el rango de `hit_score`, también suben `avg_popularity` y `hit_rate_pct`

In [19]:
hit_score_bins = (
    tracks_scored
    .withColumn(
        "hit_score_range",
        F.when(F.col("hit_score") < 40, "00-39")
         .when(F.col("hit_score") < 50, "40-49")
         .when(F.col("hit_score") < 60, "50-59")
         .when(F.col("hit_score") < 70, "60-69")
         .when(F.col("hit_score") < 80, "70-79")
         .when(F.col("hit_score") < 90, "80-89")
         .otherwise("90-100")
    )
    .groupBy("hit_score_range")
    .agg(
        F.count("*").alias("num_canciones"),
        F.round(F.avg("hit_score"), 2).alias("avg_hit_score"),
        F.round(F.avg("popularity"), 2).alias("avg_popularity"),
        F.round(F.avg("is_hit") * 100, 2).alias("hit_rate_pct")
    )
    .orderBy("hit_score_range")
)

hit_score_bins.show(truncate=False)

+---------------+-------------+-------------+--------------+------------+
|hit_score_range|num_canciones|avg_hit_score|avg_popularity|hit_rate_pct|
+---------------+-------------+-------------+--------------+------------+
|00-39          |45494        |29.33        |13.65         |0.09        |
|40-49          |52951        |45.71        |16.33         |0.27        |
|50-59          |98246        |55.31        |21.24         |0.42        |
|60-69          |123461       |65.13        |27.05         |0.75        |
|70-79          |147220       |75.23        |32.91         |1.69        |
|80-89          |117066       |83.73        |37.04         |2.77        |
|90-100         |1826         |90.93        |39.41         |2.79        |
+---------------+-------------+-------------+--------------+------------+



Los resultados muestran una tendencia positiva: conforme aumenta el rango de `Hit Score`, también aumenta la popularidad promedio de las canciones. Por ejemplo, las canciones con `Hit Score` entre 00 y 39 tienen una popularidad promedio de 13.65, mientras que las canciones con `Hit Score` entre 90 y 100 alcanzan una popularidad promedio de 39.41.

También se observa que la proporción de canciones clasificadas como hit aumenta en los rangos más altos del score. Esto sugiere que el `Hit Score` captura parcialmente un perfil acústico asociado con canciones populares.

Sin embargo, la proporción de hits sigue siendo baja incluso en los rangos más altos. Esto es esperado porque la definición de hit utilizada en el proyecto es estricta: `popularity >= 70`, umbral que corresponde aproximadamente al percentil 99 de la distribución de popularidad.

Por lo tanto, el `Hit Score` no debe interpretarse como una predicción definitiva de éxito comercial, sino como una métrica experimental que permite ordenar canciones según su similitud acústica con el perfil de canciones populares.

## Búsqueda de canciones con alto Hit Score y baja popularidad

Después de validar que el `Hit Score` tiene una relación positiva con la popularidad, se utiliza la métrica para buscar canciones poco populares pero con alto score.

Estas canciones cumplen dos condiciones:

- `popularity < 40`: canciones de baja popularidad.
- `hit_score >= 80`: canciones con un perfil acústico fuertemente alineado con el perfil de canciones populares.

Este análisis permite identificar posibles “joyas ocultas”: canciones que no destacan en popularidad dentro de Spotify, pero que comparten características acústicas con canciones hit.

Esto no significa que estas canciones necesariamente deberían ser exitosas, ya que la popularidad depende de muchos factores externos. Sin embargo, sí muestra cómo el prototipo puede usarse para explorar grandes catálogos musicales y encontrar canciones acústicamente similares a hits.

In [21]:
hidden_gems = (
    tracks_scored
    .filter(
        (F.col("popularity") < 40) &
        (F.col("hit_score") >= 80)
    )
    .select(
        "id",
        "name",
        "artists",
        "release_year",
        "popularity",
        "hit_score",
        "danceability",
        "energy",
        "loudness",
        "acousticness",
        "instrumentalness"
    )
    .orderBy(F.desc("hit_score"))
)

hidden_gems.show(50, truncate=False)

+----------------------+-------------------------------------------------------------------+-----------------------------------------------+------------+----------+---------+------------+------+--------+------------+----------------+
|id                    |name                                                               |artists                                        |release_year|popularity|hit_score|danceability|energy|loudness|acousticness|instrumentalness|
+----------------------+-------------------------------------------------------------------+-----------------------------------------------+------------+----------+---------+------------+------+--------+------------+----------------+
|3y4VWuFwpi035SOBMJeQdS|Išvažiuosiu Prie Jūros                                             |['Džordana Butkutė']                           |2001        |19        |95.15    |0.954       |0.991 |-5.768  |8.55E-4     |0.0085          |
|3O6x1B9uvMAnZJpLfzWzCM|Critical Beatdown                       

In [23]:
#Version final
columns_to_keep = [
    "id",
    "name",
    "artists",
    "id_artists",
    "release_date",
    "release_year",
    "release_decade",
    "duration_ms",
    "duration_min",
    "explicit",
    "popularity",
    "popularity_class",
    "is_hit",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "time_signature",
    "danceability_norm",
    "energy_norm",
    "loudness_norm",
    "acousticness_norm",
    "instrumentalness_norm",
    "hit_score",
    "hit_score_class"
]

tracks_scored_final = tracks_scored.select(*columns_to_keep)

tracks_scored_final.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- id_artists: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- release_decade: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- duration_min: double (nullable = true)
 |-- explicit: integer (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- popularity_class: string (nullable = true)
 |-- is_hit: integer (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-

In [24]:
tracks_scored_final.select(
    F.count("*").alias("total_filas"),
    F.min("hit_score").alias("min_hit_score"),
    F.max("hit_score").alias("max_hit_score"),
    F.round(F.avg("hit_score"), 2).alias("avg_hit_score")
).show()

+-----------+-------------+-------------+-------------+
|total_filas|min_hit_score|max_hit_score|avg_hit_score|
+-----------+-------------+-------------+-------------+
|     586264|         5.23|        95.48|        65.28|
+-----------+-------------+-------------+-------------+



**Guardamos Resultados del Hit Score**

In [25]:
tracks_scored_final.write.mode("overwrite").parquet(
    "../processed/tracks_scored.parquet"
)

26/05/04 01:35:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/04 01:35:41 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [26]:
hit_score_bins.write.mode("overwrite").parquet(
    "../processed/hit_score/hit_score_bins.parquet"
)

hidden_gems.write.mode("overwrite").parquet(
    "../processed/hit_score/hidden_gems.parquet"
)

## Conclusión del Hit Score experimental
En este notebook se construyó una primera versión del `Hit Score`, una métrica experimental basada en características acústicas asociadas con canciones populares.

El score combina variables seleccionadas durante el análisis exploratorio: `danceability`, `energy`, `loudness`, `acousticness` e `instrumentalness`.

Las variables fueron normalizadas a una escala común y posteriormente combinadas en una puntuación de 0 a 100. Las variables asociadas positivamente con popularidad aportan directamente al score, mientras que las variables asociadas negativamente se invierten.

La validación mostró una correlación positiva entre `hit_score` y `popularity`, así como un aumento de la popularidad promedio conforme aumenta el rango del score. Esto sugiere que la métrica captura parcialmente un perfil acústico asociado con canciones populares.

Sin embargo, el `Hit Score` no debe interpretarse como una predicción real de éxito comercial. La popularidad musical depende de muchos factores que no están incluidos en esta métrica, como el artista, el idioma, la letra, la promoción, las playlists y el contexto cultural.

El valor principal del `Hit Score` dentro de HitMaker AI es permitir ordenar canciones según su similitud acústica con el perfil de canciones populares y facilitar búsquedas posteriores, como la identificación de canciones poco populares con alto potencial acústico.